In [11]:
import argparse
import csv
import re
import shutil
import sys
import time
from html.parser import HTMLParser
from pathlib import Path
from urllib import robotparser
from urllib.parse import urljoin, urlparse

In [13]:
%pip install imagehash

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
%pip install requests
%pip install pillow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import requests
from PIL import Image
import imagehash

In [12]:

API = "https://commons.wikimedia.org/w/api.php"
UA = "AdireTrust-research/0.1 (hackathon research; contact: ajibuadomts@gmail.com"
DATA = Path("data")
IMG_DIR = DATA / "images"
CSV_PATH = DATA / "labels.csv"
FIELDS = [
    "image_id", "filename", "source_url", "source_page", "source_group",
    "seller_label", "license_or_tos_note", "phash",
    "label", "verified", "notes",
]
PHASH_MAX_DISTANCE = 4  # <= this many differing bits counts as a duplicate


def load_rows():
    if not CSV_PATH.exists():
        return []
    with CSV_PATH.open(newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def append_row(row):
    DATA.mkdir(exist_ok=True)
    new_file = not CSV_PATH.exists()
    with CSV_PATH.open("a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if new_file:
            w.writeheader()
        w.writerow({k: row.get(k, "") for k in FIELDS})


def next_id(rows):
    return f"img_{len(rows) + 1:05d}"


def is_duplicate(h, rows):
    for r in rows:
        if not r.get("phash"):
            continue
        if h - imagehash.hex_to_hash(r["phash"]) <= PHASH_MAX_DISTANCE:
            return r["image_id"]
    return None


def register_image(path, meta, rows, min_side=0):
    """Hash, dedupe, move into data/images, append CSV row. Returns status."""
    try:
        with Image.open(path) as im:
            if min(im.size) < min_side:
                return "skip (too small)"
            im = im.convert("RGB")
            h = imagehash.phash(im)
    except Exception as e:
        return f"skip (unreadable: {e})"
    dup = is_duplicate(h, rows)
    if dup:
        return f"skip (near-duplicate of {dup})"
    IMG_DIR.mkdir(parents=True, exist_ok=True)
    image_id = next_id(rows)
    dest = IMG_DIR / f"{image_id}{Path(path).suffix.lower() or '.jpg'}"
    if Path(path) != dest:
        shutil.copy2(path, dest)
    row = {
        "image_id": image_id,
        "filename": dest.name,
        "phash": str(h),
        "label": "",
        "verified": "no",
        **meta,
    }
    append_row(row)
    rows.append(row)
    return f"added {image_id}"


def strip_html(s):
    return re.sub(r"<[^>]+>", "", s or "").strip()


def fetch(session, url, tries=5):
    """GET with backoff on 429/503 (Wikimedia throttles bursts)."""
    for i in range(tries):
        r = session.get(url, timeout=60)
        if r.status_code in (429, 503):
            try:
                wait = int(r.headers.get("Retry-After", ""))
            except ValueError:
                wait = 5 * (2 ** i)
            print(f"  throttled ({r.status_code}), waiting {wait}s...")
            time.sleep(wait)
            continue
        r.raise_for_status()
        return r
    raise RuntimeError(f"still throttled after {tries} tries")

def cmd_commons(args):
    rows = load_rows()
    session = requests.Session()
    session.headers["User-Agent"] = UA
    params = {
        "action": "query",
        "format": "json",
        "generator": "categorymembers",
        "gcmtitle": args.category,
        "gcmtype": "file",
        "gcmlimit": 50,
        "prop": "imageinfo",
        "iiprop": "url|mime|extmetadata",
        "iiurlwidth": args.width,
    }
    seen = 0
    tmp = DATA / "_tmp"
    tmp.mkdir(parents=True, exist_ok=True)
    while True:
        r = session.get(API, params=params, timeout=30)
        r.raise_for_status()
        data = r.json()
        pages = data.get("query", {}).get("pages", {})
        for page in pages.values():
            info = (page.get("imageinfo") or [{}])[0]
            if info.get("mime") not in ("image/jpeg", "image/png"):
                continue
            seen += 1
            meta_in = info.get("extmetadata", {})
            lic = strip_html(meta_in.get("LicenseShortName", {}).get("value"))
            artist = strip_html(meta_in.get("Artist", {}).get("value"))
            desc = strip_html(meta_in.get("ImageDescription", {}).get("value"))
            note = f"{lic or 'license unknown'}; by {artist or 'unknown'}"
            if "ND" in lic.upper():
                note += "; NO-DERIVATIVES: do not crop/augment for training without permission"
            title = page.get("title", "")
            file_url = info.get("thumburl") or info.get("url")
            tmp_path = tmp / re.sub(r"[^\w.\-]", "_", title.replace("File:", ""))
            try:
                img = fetch(session, file_url)
                tmp_path.write_bytes(img.content)
            except Exception as e:
                print(f"{title}: download failed ({e})")
                continue
            status = register_image(
                tmp_path,
                {
                    "source_url": info.get("descriptionurl", file_url),
                    "source_page": info.get("descriptionurl", ""),
                    "source_group": "commons",
                    "seller_label": "",
                    "license_or_tos_note": note,
                    "notes": desc[:200],
                },
                rows,
            )
            print(f"{title}: {status}")
            tmp_path.unlink(missing_ok=True)
            time.sleep(2)  # be polite to Wikimedia
        if "continue" in data:
            params.update(data["continue"])
        else:
            break
    shutil.rmtree(tmp, ignore_errors=True)
    print(f"\nDone. {seen} candidate files seen. Now open {CSV_PATH} and fill "
          "`label` + `verified` by eye (many Commons files are people/looms, not cloth).")

_robots = {}


def allowed(url):
    """True only if the site's robots.txt lets us fetch this URL."""
    p = urlparse(url)
    base = f"{p.scheme}://{p.netloc}"
    if base not in _robots:
        rp = robotparser.RobotFileParser()
        try:
            r = requests.get(base + "/robots.txt", headers={"User-Agent": UA}, timeout=20)
        except Exception:
            return False  
        if r.status_code in (404, 410):
            rp.parse([]) 
        elif r.status_code >= 400:
            return False
        else:
            rp.parse(r.text.splitlines())
        _robots[base] = rp
    return _robots[base].can_fetch(UA, url)


class ImgCollector(HTMLParser):
    def __init__(self):
        super().__init__()
        self.urls = []

    def handle_starttag(self, tag, attrs):
        if tag != "img":
            return
        a = dict(attrs)
        cand = a.get("data-src") or a.get("data-original") or a.get("src")
        srcset = a.get("srcset") or a.get("data-srcset")
        if srcset:
            last = srcset.split(",")[-1].strip().split(" ")[0]
            if last:
                cand = last
        if cand:
            self.urls.append(cand)


def cmd_page(args):
    if not allowed(args.url):
        print("robots.txt disallows this page (or it could not be checked). Stopping.\n"
              "Ask the site owner for permission, or save images by hand and use `add`.")
        return
    rows = load_rows()
    session = requests.Session()
    session.headers["User-Agent"] = UA
    html = fetch(session, args.url).text
    col = ImgCollector()
    col.feed(html)
    urls = []
    for u in col.urls:
        if u.startswith("data:"):
            continue
        full = urljoin(args.url, u)
        if full not in urls:
            urls.append(full)
    if not urls:
        print("No <img> tags found. The page is probably JavaScript-rendered; "
              "open it in a browser and save images by hand instead.")
        return
    tmp = DATA / "_tmp"
    tmp.mkdir(parents=True, exist_ok=True)
    added = 0
    for u in urls:
        if added >= args.max:
            break
        if not allowed(u):
            print(f"{u}: skip (robots.txt)")
            continue
        try:
            resp = fetch(session, u)
        except Exception as e:
            print(f"{u}: download failed ({e})")
            continue
        if not resp.headers.get("Content-Type", "").startswith("image/"):
            continue
        name = re.sub(r"[^\w.\-]", "_", Path(urlparse(u).path).name) or "img"
        if "." not in name:
            name += ".jpg"
        tmp_path = tmp / name
        tmp_path.write_bytes(resp.content)
        status = register_image(
            tmp_path,
            {
                "source_url": u,
                "source_page": args.url,
                "source_group": args.group,
                "seller_label": args.seller_label,
                "license_or_tos_note": args.license,
                "label": args.label,
                "notes": "",
            },
            rows,
            min_side=args.min_side,
        )
        print(f"{u}: {status}")
        tmp_path.unlink(missing_ok=True)
        if status.startswith("added"):
            added += 1
        time.sleep(2)
    shutil.rmtree(tmp, ignore_errors=True)
    print(f"\nDone. {added} added. Check the page contents: logos, banners and "
          "unrelated products get picked up too. Delete those rows.")

VAM_API = "https://api.vam.ac.uk/v2/objects/search"

def cmd_vam(args):
    rows = load_rows()
    session = requests.Session()
    session.headers["User-Agent"] = UA
    tmp = DATA / "_tmp"
    tmp.mkdir(parents=True, exist_ok=True)
    added, page = 0, 1
    while added < args.max:
        r = session.get(VAM_API, params={"q": args.query, "images_exist": 1,
                                         "page_size": 50, "page": page}, timeout=30)
        r.raise_for_status()
        recs = r.json().get("records", [])
        if not recs:
            break
        for rec in recs:
            if added >= args.max:
                break
            img_id = rec.get("_primaryImageId")
            sn = rec.get("systemNumber")
            if not img_id or not sn:
                continue
            url = f"https://framemark.vam.ac.uk/collections/{img_id}/full/!1024,1024/0/default.jpg"
            title = rec.get("_primaryTitle") or rec.get("objectType", "")
            try:
                resp = fetch(session, url)
            except Exception as e:
                print(f"{sn}: download failed ({e})")
                continue
            tmp_path = tmp / f"{sn}.jpg"
            tmp_path.write_bytes(resp.content)
            status = register_image(
                tmp_path,
                {
                    "source_url": f"https://collections.vam.ac.uk/item/{sn}/",
                    "source_page": f"https://collections.vam.ac.uk/item/{sn}/",
                    "source_group": "vam",
                    "seller_label": "",
                    "license_or_tos_note": "V&A collections image; research use only, check V&A terms before sharing",
                    "notes": f"{title} {rec.get('_primaryDate', '')}".strip()[:200],
                },
                rows,
                min_side=args.min_side,
            )
            print(f"{sn} {title[:40]}: {status}")
            tmp_path.unlink(missing_ok=True)
            if status.startswith("added"):
                added += 1
            time.sleep(2)
        page += 1
    shutil.rmtree(tmp, ignore_errors=True)
    print(f"\nDone. {added} added. Search results include non-adire objects: "
          "label by eye and delete what is not cloth.")

def cmd_add(args):
    rows = load_rows()
    for p in args.paths:
        status = register_image(
            p,
            {
                "source_url": args.source_url,
                "source_page": args.source_url,
                "source_group": args.group,
                "seller_label": args.seller_label,
                "license_or_tos_note": args.license,
                "label": args.label,
                "notes": args.notes,
            },
            rows,
        )
        print(f"{p}: {status}")


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__,
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = ap.add_subparsers(dest="cmd", required=True)

    c = sub.add_parser("commons", help="pull a Wikimedia Commons category")
    c.add_argument("--category", default="Category:Adire (textile art)",
                   help="exact Commons category title (check it in a browser)")
    c.add_argument("--width", type=int, default=1024,
                   help="download width in px (smaller = faster)")
    c.set_defaults(fn=cmd_commons)

    g = sub.add_parser("page", help="download <img>s from one page (robots.txt checked)")
    g.add_argument("url")
    g.add_argument("--group", required=True)
    g.add_argument("--seller-label", default="")
    g.add_argument("--license", default="")
    g.add_argument("--label", default="")
    g.add_argument("--max", type=int, default=30)
    g.add_argument("--min-side", type=int, default=300,
                   help="skip images smaller than this many px (icons, logos)")
    g.set_defaults(fn=cmd_page)

    v = sub.add_parser("vam", help="pull V&A collection images via their API")
    v.add_argument("--query", default="adire")
    v.add_argument("--max", type=int, default=40)
    v.add_argument("--min-side", type=int, default=300)
    v.set_defaults(fn=cmd_vam)

    a = sub.add_parser("add", help="log hand-saved image(s)")
    a.add_argument("paths", nargs="+")
    a.add_argument("--source-url", required=True)
    a.add_argument("--seller-label", default="", help="what the seller/catalog claims")
    a.add_argument("--license", default="", help="license or ToS note")
    a.add_argument("--group", required=True,
                   help="source group for splitting, e.g. sokofa, vam, etsy-sellerX")
    a.add_argument("--label", default="",
                   help="your verified label if you already know it "
                        "(genuine_hand_dyed / factory_print)")
    a.add_argument("--notes", default="")
    a.set_defaults(fn=cmd_add)

    args = ap.parse_args(argv)
    args.fn(args)


if __name__ == "__main__":
    main()

  throttled (503), waiting 5s...
  throttled (503), waiting 10s...
  throttled (503), waiting 20s...
  throttled (503), waiting 40s...
  throttled (503), waiting 80s...
File:"Design for "adire" cloth" - NARA - 559019.jpg: download failed (still throttled after 5 tries)
File:Adire by Cynthia Tule-Ojochu.jpg: added img_00009
File:Adire panel.jpg: skip (near-duplicate of img_00001)
File:Adire Selection.jpg: added img_00010
File:Adire Symbols.jpg: skip (near-duplicate of img_00002)
File:Adire, Kampala.jpg: skip (near-duplicate of img_00003)
  throttled (503), waiting 5s...
  throttled (503), waiting 10s...
  throttled (503), waiting 20s...
  throttled (503), waiting 40s...
  throttled (503), waiting 80s...
File:Adire1.JPG: download failed (still throttled after 5 tries)
  throttled (503), waiting 5s...
File:Adire2.JPG: added img_00011
File:Adire3.JPG: added img_00012
File:Adire4.JPG: added img_00013
File:Adire5.JPG: added img_00014
File:Aso Adire.jpg: skip (near-duplicate of img_00004)
Fil